In [1]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv("ai4i2020.csv")   
df.head()

,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


In [3]:
X = df.drop(
    [
        "Machine failure",
        "UDI",
        "Product ID",
        "TWF",
        "HDF",
        "PWF",
        "OSF",
        "RNF"
    ],
    axis=1
)

X = pd.get_dummies(X, columns=["Type"])

y = df["Machine failure"]

print(X.shape)
print(y.shape)

(10000, 8)
(10000,)


In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [5]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

print("Training Complete")

Training Complete


In [6]:
from sklearn.metrics import accuracy_score

pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, pred))

Accuracy: 0.984


In [7]:
from sklearn.metrics import confusion_matrix, classification_report

print(confusion_matrix(y_test, pred))
print(classification_report(y_test, pred))

[[1933    6]
 [  26   35]]
              precision    recall  f1-score   support

           0       0.99      1.00      0.99      1939
           1       0.85      0.57      0.69        61

    accuracy                           0.98      2000
   macro avg       0.92      0.79      0.84      2000
weighted avg       0.98      0.98      0.98      2000



In [8]:
feature_importance = list(
    zip(X.columns, model.feature_importances_)
)

feature_importance.sort(
    key=lambda x: x[1],
    reverse=True
)

for feature, score in feature_importance:
    print(feature, score)

Torque [Nm] 0.3026209604274945
Rotational speed [rpm] 0.23659931794508648
Tool wear [min] 0.173506778304181
Air temperature [K] 0.13649186687905995
Process temperature [K] 0.1257737735440087
Type_L 0.01019634045078951
Type_M 0.009129538012543799
Type_H 0.005681424436836182


In [9]:
import joblib

joblib.dump(model, "model.pkl")
joblib.dump(X.columns.tolist(), "columns.pkl")

print("Files Saved Successfully")

Files Saved Successfully


In [10]:
import pandas as pd

importance_df = pd.DataFrame({
    "Feature": X.columns,
    "Importance": model.feature_importances_
})

importance_df.to_csv(
    "feature_importance.csv",
    index=False
)

In [11]:
import pandas as pd

importance_df = pd.DataFrame({
    "Feature": X.columns,
    "Importance": model.feature_importances_
})

print(importance_df)

print("\nSum of importances =")
print(importance_df["Importance"].sum())

                   Feature  Importance
0      Air temperature [K]    0.136492
1  Process temperature [K]    0.125774
2   Rotational speed [rpm]    0.236599
3              Torque [Nm]    0.302621
4          Tool wear [min]    0.173507
5                   Type_H    0.005681
6                   Type_L    0.010196
7                   Type_M    0.009130

Sum of importances =
1.0000000000000002
